# 01 — Data Cleaning & Preprocessing
**Logistics Operations Analytics Dashboard**

---

This notebook performs comprehensive data cleaning on the logistics orders dataset:
1. Data loading & initial inspection
2. Data type corrections
3. Missing value analysis & imputation
4. Duplicate detection & removal
5. Outlier detection (IQR method)
6. Data standardization
7. Export cleaned dataset

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Porter.in inspired color palette
COLORS = {
    'primary': '#2563eb',
    'success': '#10b981',
    'warning': '#f59e0b',
    'danger': '#ef4444',
    'dark': '#0f172a',
    'light': '#f8fafc'
}

print('Libraries loaded successfully!')

## 1. Data Loading & Initial Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('../data/logistics_orders.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Total Records: {df.shape[0]:,}')
print(f'Total Columns: {df.shape[1]}')
print(f'\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
df.head(10)

In [ ]:
# Data types and non-null counts
print('='*60)
print('COLUMN INFO')
print('='*60)
df.info()

In [ ]:
# Statistical summary
print('='*60)
print('NUMERICAL SUMMARY')
print('='*60)
df.describe().round(2)

In [ ]:
# Categorical summary
print('='*60)
print('CATEGORICAL SUMMARY')
print('='*60)
for col in df.select_dtypes(include='object').columns:
    print(f'\n{col}:')
    print(f'  Unique values: {df[col].nunique()}')
    if df[col].nunique() <= 10:
        print(df[col].value_counts().to_string())

## 2. Data Type Corrections

In [ ]:
# Convert date and time columns
df['order_date'] = pd.to_datetime(df['order_date'])
df['order_time'] = pd.to_datetime(df['order_time'], format='%H:%M:%S').dt.time

# Ensure boolean column
df['is_repeat_customer'] = df['is_repeat_customer'].astype(bool)

# Ensure numeric columns
numeric_cols = ['distance_km', 'estimated_delivery_min', 'actual_delivery_min',
                'order_value', 'delivery_fee', 'total_revenue', 'rating']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('Data types after correction:')
print(df.dtypes.to_string())

## 3. Missing Value Analysis

In [ ]:
# Missing value summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

print('='*60)
print('MISSING VALUE REPORT')
print('='*60)
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(12, 5))
cols_with_missing = missing_df[missing_df['Missing Count'] > 0].index
bars = ax.barh(cols_with_missing, missing_df.loc[cols_with_missing, 'Missing %'],
               color=COLORS['danger'], alpha=0.8, edgecolor='white')
ax.set_xlabel('Missing %', fontsize=12)
ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
ax.bar_label(bars, fmt='%.1f%%', padding=5)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Imputation Strategy ─────────────────────────────────────────────────────

# 1. actual_delivery_min: NULL for cancelled orders is VALID (expected behavior)
#    - No imputation needed — these are structurally missing
print(f'actual_delivery_min nulls match cancelled orders: '
      f'{(df["actual_delivery_min"].isnull() & (df["order_status"] == "Cancelled")).sum()}')

# 2. cancellation_reason: NULL for non-cancelled orders is VALID
#    - No imputation needed

# 3. rating: NULL for cancelled/in-transit is VALID; random nulls get median imputation
rating_mask = df['rating'].isnull() & (df['order_status'].isin(['Delivered', 'Returned']))
median_rating = df.loc[df['order_status'] == 'Delivered', 'rating'].median()
df.loc[rating_mask, 'rating'] = median_rating
print(f'Imputed {rating_mask.sum()} rating values with median: {median_rating}')

# 4. distance_km: Impute with median distance for same vehicle type
dist_nulls = df['distance_km'].isnull().sum()
df['distance_km'] = df.groupby('vehicle_type')['distance_km'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Imputed {dist_nulls} distance_km values with vehicle-type median')

# 5. delivery_fee: Impute with median fee for same vehicle type
fee_nulls = df['delivery_fee'].isnull().sum()
df['delivery_fee'] = df.groupby('vehicle_type')['delivery_fee'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Imputed {fee_nulls} delivery_fee values with vehicle-type median')

# Recalculate total_revenue where delivery_fee was imputed
mask_recalc = (df['order_status'] != 'Cancelled') & (df['total_revenue'] == 0)
df.loc[mask_recalc, 'total_revenue'] = df.loc[mask_recalc, 'order_value'] + df.loc[mask_recalc, 'delivery_fee']

print('\nPost-imputation missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 4. Duplicate Detection

In [ ]:
# Check for duplicate order IDs
dup_orders = df['order_id'].duplicated().sum()
print(f'Duplicate order IDs: {dup_orders}')

# Check for full row duplicates
dup_rows = df.duplicated().sum()
print(f'Duplicate rows: {dup_rows}')

if dup_rows > 0:
    print(f'Removing {dup_rows} duplicate rows...')
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'New shape: {df.shape}')
else:
    print('No duplicates found. Data is clean.')

## 5. Outlier Detection (IQR Method)

In [ ]:
def detect_outliers_iqr(series, multiplier=1.5):
    """Detect outliers using the IQR method."""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    outliers = (series < lower) | (series > upper)
    return outliers, lower, upper

outlier_cols = ['order_value', 'delivery_fee', 'distance_km', 'actual_delivery_min']

print('='*70)
print(f'{"Column":<25} {"Outliers":<10} {"% of Data":<12} {"Lower":<12} {"Upper":<12}')
print('='*70)

for col in outlier_cols:
    valid = df[col].dropna()
    outliers, lower, upper = detect_outliers_iqr(valid)
    n_outliers = outliers.sum()
    pct = (n_outliers / len(valid)) * 100
    print(f'{col:<25} {n_outliers:<10} {pct:<12.2f} {lower:<12.2f} {upper:<12.2f}')

In [ ]:
# Box plots for outlier visualization
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for i, col in enumerate(outlier_cols):
    valid = df[col].dropna()
    bp = axes[i].boxplot(valid, patch_artist=True,
                         boxprops=dict(facecolor=COLORS['primary'], alpha=0.6),
                         medianprops=dict(color=COLORS['danger'], linewidth=2),
                         whiskerprops=dict(color=COLORS['dark']),
                         capprops=dict(color=COLORS['dark']))
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].grid(axis='y', alpha=0.3)

fig.suptitle('Outlier Detection — Box Plots', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cap outliers (winsorization) rather than removing — preserves data volume
def cap_outliers(df, col, multiplier=3.0):
    """Cap outliers using IQR method with a generous multiplier."""
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    after = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f'{col}: Capped {before} outliers (range: {lower:.1f} - {upper:.1f})')
    return df

for col in ['order_value', 'distance_km']:
    df = cap_outliers(df, col)

print('\nOutlier capping complete.')

## 6. Data Standardization & Validation

In [ ]:
# Standardize city names (trim whitespace, title case)
df['city'] = df['city'].str.strip().str.title()
df['pickup_area'] = df['pickup_area'].str.strip().str.title()
df['delivery_area'] = df['delivery_area'].str.strip().str.title()

# Validate order_status values
valid_statuses = ['Delivered', 'Cancelled', 'In-Transit', 'Returned']
invalid_status = ~df['order_status'].isin(valid_statuses)
print(f'Invalid order statuses: {invalid_status.sum()}')

# Validate vehicle_type values
valid_vehicles = ['Bike', 'Auto', 'Mini Truck', 'Truck']
invalid_vehicle = ~df['vehicle_type'].isin(valid_vehicles)
print(f'Invalid vehicle types: {invalid_vehicle.sum()}')

# Validate rating range
invalid_rating = df['rating'].notna() & ((df['rating'] < 1) | (df['rating'] > 5))
print(f'Invalid ratings (outside 1-5): {invalid_rating.sum()}')

# Validate non-negative values
for col in ['order_value', 'delivery_fee', 'distance_km']:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f'WARNING: {col} has {neg_count} negative values!')
        df.loc[df[col] < 0, col] = df[col].median()

print('\nValidation complete!')

In [ ]:
# Final data quality summary
print('='*60)
print('CLEANED DATASET SUMMARY')
print('='*60)
print(f'Total Records:     {len(df):,}')
print(f'Total Columns:     {df.shape[1]}')
print(f'Unique Customers:  {df["customer_id"].nunique():,}')
print(f'Unique Drivers:    {df["driver_id"].nunique()}')
print(f'Unique Cities:     {df["city"].nunique()}')
print(f'Date Range:        {df["order_date"].min().date()} to {df["order_date"].max().date()}')
print(f'\nRemaining Missing Values:')
remaining = df.isnull().sum()
print(remaining[remaining > 0].to_string() if remaining.sum() > 0 else '  None (expected nulls only)')

## 7. Export Cleaned Dataset

In [ ]:
# Save cleaned dataset
output_path = '../data/logistics_orders_cleaned.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset saved to: {output_path}')
print(f'Shape: {df.shape}')
print(f'File size: {pd.io.common.file_exists(output_path)}')
print('\nData cleaning pipeline complete!')